# Modern CNN Architectures

**Companion lesson:** https://ml-viz.vercel.app/courses/cnns/04-modern-architectures

From-scratch parameter counting, bottleneck blocks, and residual connections — the building blocks of every modern vision model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## 1×1 convolutions: channel mixing without spatial mixing

A 1×1 conv applies the same linear projection to every spatial position independently.
It is the primary tool for reducing or expanding channel counts cheaply.

In [ ]:
def conv1x1_params(c_in, c_out):
    """Parameters of a 1×1 convolution (no bias)."""
    return 1 * 1 * c_in * c_out

def conv3x3_params(c_in, c_out):
    """Parameters of a 3×3 convolution (no bias)."""
    return 3 * 3 * c_in * c_out

# Standard 3×3: 256 → 256
standard = conv3x3_params(256, 256)
print(f'Standard 3×3 (256→256): {standard:,} params')

# Bottleneck: 1×1 (256→64) + 3×3 (64→64) + 1×1 (64→256)
bottleneck = conv1x1_params(256, 64) + conv3x3_params(64, 64) + conv1x1_params(64, 256)
print(f'Bottleneck  (256→64→64→256): {bottleneck:,} params')
print(f'Reduction factor: {standard/bottleneck:.1f}×')

assert standard == 589_824
assert bottleneck == 69_632
print(f'VERIFY: bottleneck uses {standard//bottleneck}× fewer parameters.')

## Bottleneck residual block — numpy implementation

Implement the forward pass (no training, just shape verification).
The skip connection adds the input back before the final ReLU.

In [ ]:
def relu(x): return np.maximum(0, x)

def layernorm_simple(x):
    """Simplified normalisation: standardise over the channel axis."""
    mu  = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True) + 1e-5
    return (x - mu) / std

def conv_nd(x, W):
    """Simple matrix-multiply proxy for a 1×1 conv on a batch of vectors.
    x: (N, C_in), W: (C_in, C_out) → output: (N, C_out)."""
    return x @ W

np.random.seed(7)
N, C = 8, 256   # batch of 8 spatial positions, 256 channels
x = np.random.randn(N, C) * 0.1

# Bottleneck weights
scale = 0.1
W_down = np.random.randn(256, 64)  * scale
W_3x3  = np.random.randn(64, 64)   * scale
W_up   = np.random.randn(64, 256)  * scale

def bottleneck_block(x, W_down, W_3x3, W_up):
    """Bottleneck residual block (1×1 → 3×3 → 1×1 + skip)."""
    h = relu(layernorm_simple(conv_nd(x, W_down)))   # 256 → 64
    h = relu(layernorm_simple(conv_nd(h, W_3x3)))    # 64  → 64
    h = layernorm_simple(conv_nd(h, W_up))           # 64  → 256
    return relu(x + h)                               # skip + ReLU

out = bottleneck_block(x, W_down, W_3x3, W_up)
print('Input shape:', x.shape, '→ Output shape:', out.shape)
assert out.shape == x.shape, 'Bottleneck must preserve shape'
print('VERIFY: bottleneck block input == output shape.')

## Skip connections and gradient flow

With y = F(x) + x, the gradient ∂y/∂x = ∂F/∂x + 1.
The constant 1 is the skip path — gradients always flow regardless of F.

In [ ]:
# Compare activation norms through deep stacks: plain vs. residual
np.random.seed(1)
n_blocks = 30
d = 64

plain_norms = []
residual_norms = []

x_plain = np.random.randn(1, d)
x_res   = np.random.randn(1, d)

for _ in range(n_blocks):
    W = np.random.randn(d, d) * (0.1 / np.sqrt(d))
    # Plain: x → tanh(Wx)
    x_plain = np.tanh(x_plain @ W)
    plain_norms.append(np.linalg.norm(x_plain))
    # Residual: x → x + tanh(Wx)
    x_res = x_res + np.tanh(x_res @ W)
    residual_norms.append(np.linalg.norm(x_res))

fig, ax = plt.subplots()
ax.plot(plain_norms, color='#f59e0b', label='plain (tanh layers)')
ax.plot(residual_norms, color='#6366f1', label='residual')
ax.set_xlabel('block depth'); ax.set_ylabel('||activations||')
ax.set_title('Activation norm through 30 blocks: plain vs. residual')
ax.legend(); plt.show()

print('Final plain norm:', round(plain_norms[-1], 4))
print('Final residual norm:', round(residual_norms[-1], 4))

## EfficientNet compound scaling

Compound scaling jointly increases depth $d=\alpha^\phi$, width $w=\beta^\phi$, resolution $r=\gamma^\phi$
subject to $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$ (doubling FLOPs per $\phi$ step).

In [ ]:
# EfficientNet compound scaling: FLOPs grow as w^2 * d * r^2
# (channels^2 for conv, linear in depth, resolution^2 for spatial)
def flops_scale(alpha, beta, gamma, phi):
    d = alpha ** phi
    w = beta  ** phi
    r = gamma ** phi
    return w**2 * d * r**2   # proportional to total FLOPs

# EfficientNet-found constants
alpha, beta, gamma = 1.2, 1.1, 1.15

print('phi | depth | width | res  | rel. FLOPs')
base = flops_scale(alpha, beta, gamma, 0)
for phi in range(8):
    f = flops_scale(alpha, beta, gamma, phi)
    print(f'{phi}   | {alpha**phi:.2f}  | {beta**phi:.2f}  | {gamma**phi:.2f} | {f/base:.1f}×')

# Verify: each step approximately doubles FLOPs
ratios = [flops_scale(alpha, beta, gamma, phi+1) / flops_scale(alpha, beta, gamma, phi)
          for phi in range(6)]
assert all(abs(r - 2.0) < 0.2 for r in ratios), 'Each phi step should ≈ double FLOPs'
print('VERIFY: each compound scaling step doubles FLOPs.')

## Depthwise separable convolutions

MobileNet factorizes standard convolution into depthwise (spatial) + pointwise (channel) steps.
This reduces parameters and FLOPs by ~8× for typical channel counts.

In [ ]:
def compare_params(K, C_in, C_out):
    """Compare standard vs. depthwise-separable conv parameters."""
    standard   = K * K * C_in * C_out
    depthwise  = K * K * C_in           # one K×K filter per input channel
    pointwise  = 1 * 1 * C_in * C_out  # 1×1 to mix channels
    dw_sep_total = depthwise + pointwise
    return standard, dw_sep_total, standard / dw_sep_total

print(f'{"Channels":>8} | {"Standard":>12} | {"DW-Sep":>10} | {"Reduction":>10}')
for C in [32, 64, 128, 256, 512]:
    std, dws, ratio = compare_params(K=3, C_in=C, C_out=C)
    print(f'{C:>8} | {std:>12,} | {dws:>10,} | {ratio:>9.1f}×')

# For large C, reduction approaches K^2 = 9 (for K=3)
std, dws, ratio = compare_params(K=3, C_in=1000, C_out=1000)
assert ratio > 8.5, 'For large C, DW-sep should give >8.5× reduction'
print('VERIFY: depthwise separable ≈ 9× fewer params for large C.')

## Key takeaways

- **1×1 convolutions** mix channels without spatial processing — core tool for channel count control.
- **Bottleneck block** (1×1 → 3×3 → 1×1) gives ~8-17× parameter savings vs. naive 3×3.
- **Skip connections**: gradient = ∂F/∂x + 1 — the +1 keeps gradients flowing regardless of depth.
- **MobileNet** depthwise separable conv: ~8× fewer params for the same receptive field.
- **EfficientNet** jointly scales depth, width, resolution to optimise the accuracy/FLOP tradeoff.

## ✏️ Your turn

### Exercise 1 — 1×1 convolution forward pass

A 1×1 convolution with weight matrix $W \in \mathbb{R}^{C_{in} \times C_{out}}$ applies the same linear projection to every spatial position. Implement it and verify: shape, and that it equals a per-position matrix multiply.

In [ ]:
import numpy as np

def conv1x1_forward(x, W):
    """1×1 convolution forward pass.
    x: (H, W, C_in) feature map, W: (C_in, C_out) weight matrix.
    Returns: (H, W, C_out) output."""
    # TODO(you): apply W to the channel dimension at every spatial position
    # Hint: reshape x to (H*W, C_in), apply W, reshape back
    ...

In [ ]:
np.random.seed(0)
H, W_size, C_in, C_out = 4, 4, 8, 3
x_feat = np.random.randn(H, W_size, C_in)
W_proj = np.random.randn(C_in, C_out)

out = conv1x1_forward(x_feat, W_proj)

assert out.shape == (H, W_size, C_out), \
    f"output shape must be ({H}, {W_size}, {C_out}), got {out.shape}"

# Spot-check: output at position (0,0) equals x[0,0] @ W
assert np.allclose(out[0, 0], x_feat[0, 0] @ W_proj), \
    "output at (0,0) must equal x[0,0] @ W"
assert np.allclose(out[2, 3], x_feat[2, 3] @ W_proj), \
    "output at (2,3) must equal x[2,3] @ W (same W at every position)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def conv1x1_forward(x, W):
    H, W_size, C_in = x.shape
    return (x.reshape(-1, C_in) @ W).reshape(H, W_size, -1)
```

</details>

### Exercise 2 — Bottleneck parameter count

Compute the total number of parameters in a bottleneck residual block with channels $C_{in} = C_{out} = 256$ and bottleneck width $B = 64$. Compare to a plain two-layer 3×3 block.

In [ ]:
def bottleneck_params(c_in, c_out, b):
    """Parameter count for a bottleneck residual block (ignoring bias):
    1×1 (c_in→b) + 3×3 (b→b) + 1×1 (b→c_out)."""
    # TODO(you): sum the three layer parameter counts
    ...

def plain_block_params(c):
    """Parameter count for two stacked 3×3 convs (c→c→c), ignoring bias."""
    # TODO(you): two 3×3 convolutions
    ...

In [ ]:
bt = bottleneck_params(256, 256, 64)
pl = plain_block_params(256)

assert bt == 69_632, \
    f"bottleneck params should be 69,632, got {bt}"
assert pl == 1_179_648, \
    f"plain block params should be 1,179,648, got {pl}"
assert pl // bt >= 16, \
    "plain block should have at least 16× more parameters than bottleneck"
print(f"Bottleneck: {bt:,} params")
print(f"Plain:      {pl:,} params ({pl//bt}× more)")
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bottleneck_params(c_in, c_out, b):
    return (1*1*c_in*b) + (3*3*b*b) + (1*1*b*c_out)

def plain_block_params(c):
    return 2 * (3*3*c*c)
```

</details>

### Exercise 3 — Residual block gradient

For y = F(x) + x, compute ∂L/∂x given ∂L/∂y (the upstream gradient).
Verify the skip path carries the gradient unchanged even when F's Jacobian is zero.

In [ ]:
import numpy as np

def residual_backward(grad_y, jac_F):
    """Backward pass of y = F(x) + x.
    grad_y: upstream gradient ∂L/∂y (1-D array).
    jac_F: Jacobian ∂F/∂x (2-D square matrix).
    Returns ∂L/∂x."""
    # TODO(you): ∂L/∂x = grad_y @ (jac_F + I)
    ...

In [ ]:
d = 4
grad_y = np.array([1.0, 2.0, 0.5, -1.0])
jac_F  = np.random.randn(d, d) * 0.1

grad_x = residual_backward(grad_y, jac_F)

assert grad_x.shape == grad_y.shape, "gradient shape must match input shape"
assert np.allclose(grad_x, grad_y @ (jac_F + np.eye(d))), \
    "∂L/∂x must equal grad_y @ (∂F/∂x + I)"

# Key property: when F's Jacobian is zero, skip path carries full gradient
zero_jac = np.zeros((d, d))
grad_x_dead = residual_backward(grad_y, zero_jac)
assert np.allclose(grad_x_dead, grad_y), \
    "with zero Jacobian (dead block), skip path must pass gradient unchanged"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def residual_backward(grad_y, jac_F):
    return grad_y @ (jac_F + np.eye(len(grad_y)))
```

</details>